# Speech Translation

In [ ]:
!pip install googletrans==4.0.0-rc1

In [ ]:
# implementing for text basis....

# Import the necessary module
from googletrans import Translator

# Initialize the translator
translator = Translator()

# Supported languages dictionary
languages = {
    'en': 'English',
    'hi': 'Hindi',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'zh-cn': 'Chinese (Simplified)',
    'ar': 'Arabic',
    'ja': 'Japanese'
}

# Default source language (user's spoken language)
default_src_lang = 'hi'  # Adapt this based on the user's language preference

# Function to translate text
def translate_text():
    # Ask user for input text
    text = input(f"Enter the text you want to translate from {languages[default_src_lang]}: ")

    # Display the supported languages
    print("Supported languages:")
    for code, lang in languages.items():
        print(f"{code}: {lang}")

    # Ask user for target language, default to English
    dest_lang = input("Enter the target language code (default is 'en' for English): ").strip().lower() or 'en'

    if dest_lang not in languages:
        print("Invalid target language code.")
        return

    # Perform translation
    translated = translator.translate(text, src=default_src_lang, dest=dest_lang).text
    print(f"Translation from {languages[default_src_lang]} to {languages[dest_lang]}: {translated}")

# Run the function
translate_text()


In [ ]:
!pip install gtts googletrans==4.0.0-rc1 speechrecognition pydub

In [ ]:
!pip install sounddevice googletrans==4.0.0-rc1 gtts speechrecognition pydub

In [ ]:
!pip install sounddevice wavio SpeechRecognition

In [ ]:
# implementing for voice basis....

import sounddevice as sd
import wavio
import speech_recognition as sr
from googletrans import Translator
from gtts import gTTS
from pydub import AudioSegment
from pydub.playback import play
import os

# Initialize the translator
translator = Translator()

# Supported languages dictionary
languages = {
    'en': 'English',
    'hi': 'Hindi',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'zh-cn': 'Chinese (Simplified)',
    'ar': 'Arabic',
    'ja': 'Japanese'
}

# Default source language (user's spoken language)
default_src_lang = 'hi'  # Adapt this based on the user's language preference
default_dest_lang = 'en'  # Default translation target language is English

# Function to record audio
def record_audio(filename, duration=5, fs=16000):
    print(f"Recording for {duration} seconds...")
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()  # Wait until the recording is finished
    wavio.write(filename, recording, fs, sampwidth=2)
    print("Recording complete.")

# Function to recognize speech and translate
def recognize_and_translate():
    audio_filename = "./test-recording/input/input.wav"
    
    # Record audio from the microphone
    record_audio(audio_filename)
    
    recognizer = sr.Recognizer()
    
    with sr.AudioFile(audio_filename) as source:
        audio = recognizer.record(source)
        
        try:
            # Recognize speech using Google's speech recognition
            text = recognizer.recognize_google(audio, language=default_src_lang)
            print(f"Recognized text: {text}")
            
            # Translate the recognized text
            translated_text = translator.translate(text, src=default_src_lang, dest=default_dest_lang).text
            print(f"Translation to {languages[default_dest_lang]}: {translated_text}")
            
            # Convert translated text to speech
            tts = gTTS(translated_text, lang=default_dest_lang)
            tts.save("translated.mp3")
            
            # Play the translated speech
            translated_audio = AudioSegment.from_mp3("./test-recording/translated/translated.mp3")
            play(translated_audio)
            os.remove("translated.mp3")
            
        except sr.UnknownValueError:
            print("Could not understand the audio.")
        except sr.RequestError as e:
            print(f"Error with the speech recognition service; {e}")

# Run the function
recognize_and_translate()


In [ ]:
pip install langdetect

In [ ]:
pip install sounddevice wavio SpeechRecognition googletrans==4.0.0-rc1 langdetect gtts pydub

In [ ]:
import sounddevice as sd
import wavio
import speech_recognition as sr
from googletrans import Translator
from gtts import gTTS
from pydub import AudioSegment
from pydub.playback import play
import os
import langdetect

# Initialize the translator
translator = Translator()

# Supported languages dictionary
languages = {
    'en': 'English',
    'hi': 'Hindi',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'zh-cn': 'Chinese (Simplified)',
    'ar': 'Arabic',
    'ja': 'Japanese'
}

# Function to record audio
def record_audio(filename, duration=5, fs=44100):
    print(f"Recording for {duration} seconds...")
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()
    wavio.write(filename, recording, fs, sampwidth=2)
    print("Recording complete.")

# Function to recognize speech and detect language
def recognize_speech(audio_file):
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_file) as source:
        audio = recognizer.record(source)
        try:
            text = recognizer.recognize_google(audio)
            lang = langdetect.detect(text)
            return text, lang
        except sr.UnknownValueError:
            print("Could not understand the audio.")
            return None, None
        except sr.RequestError as e:
            print(f"Error with the speech recognition service; {e}")
            return None, None

# Function to translate text
def translate_text(text, src_lang, dest_lang):
    translated = translator.translate(text, src=src_lang, dest=dest_lang)
    return translated.text

# Function to convert text to speech and play it
def text_to_speech(text, lang):
    tts = gTTS(text, lang=lang)
    tts.save("./test_recording/output.mp3")
    audio = AudioSegment.from_mp3("./test_recording/output.mp3")
    play(audio)
    os.remove("./test_recording/output.mp3")

# Main function to handle the conversation
def two_way_translation():
    user1_lang = None
    user2_lang = None

    while True:
        # User 1's turn
        print("User 1, please speak:")
        record_audio("./test_recording/user1_input.wav")
        user1_text, detected_lang1 = recognize_speech("./test_recording/user1_input.wav")
        
        if user1_text:
            if not user1_lang:
                user1_lang = detected_lang1
                print(f"User 1's language detected as: {languages.get(user1_lang, user1_lang)}")
            
            print(f"User 1 said: {user1_text}")
            
            if user2_lang:
                translated_text = translate_text(user1_text, user1_lang, user2_lang)
                print(f"Translated for User 2: {translated_text}")
                text_to_speech(translated_text, user2_lang)
        
        # User 2's turn
        print("User 2, please speak:")
        record_audio("./test_recording/user2_input.wav")
        user2_text, detected_lang2 = recognize_speech("./test_recording/user2_input.wav")
        
        if user2_text:
            if not user2_lang:
                user2_lang = detected_lang2
                print(f"User 2's language detected as: {languages.get(user2_lang, user2_lang)}")
            
            print(f"User 2 said: {user2_text}")
            
            if user1_lang:
                translated_text = translate_text(user2_text, user2_lang, user1_lang)
                print(f"Translated for User 1: {translated_text}")
                text_to_speech(translated_text, user1_lang)

        # Ask if users want to continue
        continue_conversation = input("Do you want to continue the conversation? (yes/no): ").lower()
        if continue_conversation != 'yes':
            break

# Run the function
two_way_translation()